# Event Chain Extraction with Embedding Clustering

### Overview

runs after notebook 1 (cleaning). Reads `data/stylecom_cleaned.csv` for the original review text.

Input: `stylecom_cleaned.csv`, `cluster_labels.csv` (manually written in google sheets)

Outputs:
- `event_chains/triples_raw.jsonl`: raw triples
- `event_chains/event_type_map.json`: verb lemma assigned to cluster label
- `event_chains/domain_verb_diagnostics.csv`: poor-fit domain verbs for review
- `checkpoints/event_sequences.jsonl`: final per-review event sequences

Pipeline:
- **1. Setup:** imports and checkpoint
- **2. NLP preprocessing with SpaCy & extracting triples:** Processes the dataset again, this time not only nouns, and extracts triples
- **3. Abstract verbs via embedding clustering:** Find optimal K and use sentence-transformers/all-MiniLM-L6-v2 to cluster verbs
- **4. Verb cluster diagnostics:** check verbs distance from centroid
- **5. Per-review event sequences:** Build a jsonl file of seuqences per review for further processing
- **6. Sanity checks:** Check overall distribution of assignments and short event sequences





## 1. Setup

In [1]:
# Cell 1: Imports
import json
import re
from collections import Counter
from itertools import pairwise
from pathlib import Path

import numpy as np
import pandas as pd
import spacy
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sentence_transformers import SentenceTransformer

/Users/zoeoggel/Data Science/Thesis/.ths/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
# Cell 2: Paths
DATA_PATH        = Path("data/stylecom_cleaned.csv")
OUT_DIR          = Path("event_chains")
OUT_DIR.mkdir(exist_ok=True)
CLUSTER_LABELS_PATH = Path("data/cluster_labels.csv")

TRIPLES_PATH     = OUT_DIR / "triples_raw.jsonl"
MAP_PATH         = OUT_DIR / "event_type_map.json"
DIAG_PATH        = OUT_DIR / "domain_verb_diagnostics.csv"
SEQUENCE_PATH         = "checkpoints/event_sequences.jsonl"

## 2. NLP preprocessing with SpaCy & extracting triples

In [3]:
# Cell 4: Load data (use noun tokens dataframe)
df = pd.read_csv(DATA_PATH)

# Add doc_id
def assign_doc_id(df):
      if "doc_id" in df.columns:
          return df
      return df.reset_index(names="doc_id")

df = assign_doc_id(df)

print(f"Loaded {len(df)} reviews")
print("Columns:", df.columns.tolist())

Loaded 6501 reviews
Columns: ['doc_id', 'year', 'season', 'designer', 'author', 'city', 'date', 'review']


In [4]:
# Cell 5: Load spaCy, full pipeline
try:
    nlp = spacy.load("en_core_web_lg")
except OSError:
    raise OSError(
        "spaCy model not installed. Run:\n"
        "  python -m spacy download en_core_web_lg"
    )

nlp.max_length = max(2_000_000, nlp.max_length)
print("Loaded en_core_web_lg", nlp.pipe_names)

Loaded en_core_web_lg ['tok2vec', 'tagger', 'parser', 'attribute_ruler', 'lemmatizer', 'ner']


In [5]:
# Cell 6: Helper function for extracting triples

def extract_triples_from_doc(doc, sentence_offset=0):
    """
    Extract triples from reviews, take passive verbs into account
    Lemmatize, 
    """
    triples = []
    for sent_idx, sent in enumerate(doc.sents):
        abs_sent_idx = sentence_offset + sent_idx
        for token in sent:
            if token.pos_ not in ("VERB", "AUX"):
                continue
            if token.dep_ in {"aux", "cop", "auxpass"}:
                continue

            verb_lemma = token.lemma_.lower()
            children = list(token.children)
            child_deps = {c.dep_: c for c in children}

            is_passive = (
                "nsubjpass" in child_deps
                or "auxpass" in child_deps
            )

            if is_passive:
                subj_token = child_deps.get("nsubjpass")
                subj = subj_token.lemma_.lower() if subj_token else None
                agent_token = child_deps.get("agent")
                if agent_token:
                    agent_children = {c.dep_: c for c in agent_token.children}
                    pobj = agent_children.get("pobj")
                    obj = pobj.lemma_.lower() if pobj else agent_token.lemma_.lower()
                else:
                    obj = None
            else:
                subj_token = child_deps.get("nsubj")
                subj = subj_token.lemma_.lower() if subj_token else None
                dobj_token = child_deps.get("dobj")
                obj = dobj_token.lemma_.lower() if dobj_token else None

            if subj is None:
                continue

            triples.append({
                "subject": subj,
                "verb": verb_lemma,
                "object": obj,
                "sentence_idx": abs_sent_idx,
                "is_passive": is_passive,
            })
    return triples

In [6]:
# Cell 7: Run spaCy over all reviews and save triples_raw.jsonl, uses original review text

BATCH_SIZE = 64
texts = df["review"].fillna("").tolist()
columns = ['doc_id', 'year', 'season', 'designer', 'author', 'city', 'date', 'review']
df_spacy = df[columns].copy()

# Run full nlp pipeline on texts to normalize again
texts = nlp.pipe(texts, batch_size=BATCH_SIZE)

triples = []
for (_, row), doc in zip(df_spacy.iterrows(), texts):
    doc_triples = extract_triples_from_doc(doc)
    triples.append({
        "review_id": int(row["doc_id"]),
        "brand":     str(row["designer"]),
        "year":      int(row["year"]) if not pd.isna(row["year"]) else None,
        "season":    str(row["season"]),
        "author":    str(row["author"]),
        "city":      str(row["city"]),
        "date":      str(row["date"]),
        "triples":   doc_triples,
    })

with open(TRIPLES_PATH, "w") as f:
    for r in triples:
        f.write(json.dumps(r) + "\n")

total_triples = sum(len(r["triples"]) for r in triples)
print(f"Saved {len(triples)} reviews, {total_triples:,} tirples at {TRIPLES_PATH}")

Saved 6501 reviews, 119,370 tirples at event_chains/triples_raw.jsonl


In [7]:
# Cell 8: Inspection

sample = triples[1]
print(f"Review {sample['review_id']}: {sample['brand']} {sample['season']} {sample['year']}")
print(f"  {len(sample['triples'])} triples")
for t in sample["triples"][:6]:
    print(f"  {t}")

Review 1: Giorgio Armani Spring 2000
  13 triples
  {'subject': 'armani', 'verb': 'propose', 'object': 'silhouette', 'sentence_idx': 0, 'is_passive': False}
  {'subject': 'foam', 'verb': 'infuse', 'object': 'collection', 'sentence_idx': 1, 'is_passive': False}
  {'subject': 'woman', 'verb': 'be', 'object': None, 'sentence_idx': 2, 'is_passive': False}
  {'subject': 'she', 'verb': 'be', 'object': None, 'sentence_idx': 2, 'is_passive': False}
  {'subject': 'innovation', 'verb': 'come', 'object': None, 'sentence_idx': 3, 'is_passive': False}
  {'subject': 'model', 'verb': 'wear', 'object': 'piece', 'sentence_idx': 3, 'is_passive': False}


## 3. Abstract verbs via embedding clustering

In [8]:
# Cell 9: Collect verb lemmas and their corpus frequencies

verb_counter = Counter()
for r in triples:
    for t in r["triples"]:
        verb_counter[t["verb"]] += 1

unique_verbs = sorted(verb_counter.keys())
print(f"{len(unique_verbs)} unique verbs, {sum(verb_counter.values()):,} total tokens")

3309 unique verbs, 119,370 total tokens


In [9]:
# Cell 10: Compute verb embeddings with sentence-transformers

MODEL_NAME = "sentence-transformers/all-MiniLM-L6-v2"
st_model = SentenceTransformer(MODEL_NAME)

verb_embeddings = st_model.encode(unique_verbs, show_progress_bar=True,
                                   batch_size=128, normalize_embeddings=True)
print(f"Embeddings shape: {verb_embeddings.shape}")

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10978.18it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Batches: 100%|██████████| 26/26 [00:00<00:00, 31.04it/s]

Embeddings shape: (3309, 384)


In [10]:
# Cell 11: Sweep k and select by silhouette score,  
# currently fixs at k=30 due to flat silhuouette score

k_values = [15, 20, 25, 30, 35, 40, 45, 50, 55, 60, 65, 70, 75, 80, 85, 90, 95, 100]
results = {}

for k in k_values:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(verb_embeddings)
    sil = silhouette_score(verb_embeddings, labels, sample_size=min(5000, len(unique_verbs)), metric="cosine")
    results[k] = {"model": km, "labels": labels, "silhouette": sil}
    print(f"k={k:2d}  silhouette={sil:.4f}")

best_sil_k = max(results, key=lambda k: results[k]["silhouette"])
print(f"\nBest silhouette at k={best_sil_k} (for reference only)")

best_k      = 30
best_km     = results[best_k]["model"]
best_labels = results[best_k]["labels"]
print(f"Using k={best_k} (fixed)")

k=15  silhouette=0.0296
k=20  silhouette=0.0342
k=25  silhouette=0.0341
k=30  silhouette=0.0367
k=35  silhouette=0.0400
k=40  silhouette=0.0387
k=45  silhouette=0.0390
k=50  silhouette=0.0423
k=55  silhouette=0.0396
k=60  silhouette=0.0417
k=65  silhouette=0.0412
k=70  silhouette=0.0429
k=75  silhouette=0.0428
k=80  silhouette=0.0449
k=85  silhouette=0.0424
k=90  silhouette=0.0461
k=95  silhouette=0.0483
k=100  silhouette=0.0453

Best silhouette at k=95 (for reference only)
Using k=30 (fixed)


In [11]:
# Cell 12: Print top-10 verbs per cluster for manual labeling (done in a google sheet saved to repo as cluster_labels.csv)

cluster_verbs = {}
for verb, label in zip(unique_verbs, best_labels):
    cluster_verbs.setdefault(int(label), []).append(verb)

print(f"Cluster contents (k={best_k})")

for cluster_id in sorted(cluster_verbs):
    top = sorted(cluster_verbs[cluster_id],
                 key=lambda v: verb_counter[v], reverse=True)[:10]
    freq_total = sum(verb_counter[v] for v in cluster_verbs[cluster_id])
    print(f"Cluster {cluster_id}  (n={len(cluster_verbs[cluster_id])}, freq={freq_total})")
    print("  Top verbs:", ", ".join(top))
    print()

Cluster contents (k=30)
Cluster 0  (n=78, freq=407)
  Top verbs: indulge, ensure, precede, prevail, dictate, assure, populate, conceive, reproduce, anticipate

Cluster 1  (n=56, freq=522)
  Top verbs: change, replace, maintain, merge, tweak, reconfigure, alter, deploy, transpire, copy

Cluster 2  (n=130, freq=874)
  Top verbs: bear, evolve, head, whip, steer, slash, sling, stray, toy, stud

Cluster 3  (n=98, freq=745)
  Top verbs: evoke, embellish, inject, elevate, embody, envision, exude, accentuate, encapsulate, inset

Cluster 4  (n=79, freq=1340)
  Top verbs: focus, reveal, reflect, aim, resemble, expose, flare, peek, shine, flash

Cluster 5  (n=124, freq=2144)
  Top verbs: design, create, build, draw, layer, print, form, base, line, shape

Cluster 6  (n=113, freq=791)
  Top verbs: separate, drop, strip, toss, abandon, explode, split, fold, dissolve, disappear

Cluster 7  (n=102, freq=947)
  Top verbs: mix, paint, matter, fill, concentrate, flow, wash, absorb, adorn, spill

Cluster 

In [12]:
# Cell 13: Build verb to cluster label map from data/cluster_labels.csv and save

cluster_label_df = pd.read_csv(CLUSTER_LABELS_PATH)

# Build dict: cluster integer ID → Original Description label
CLUSTER_LABELS = dict(
    zip(cluster_label_df["Cluster"].astype(int),
        cluster_label_df["Original Description"])
)

print(f"Loaded {len(CLUSTER_LABELS)} cluster labels from {CLUSTER_LABELS_PATH}")


# Build verb to label mapping
event_type_map = {
    verb: CLUSTER_LABELS[int(label)]
    for verb, label in zip(unique_verbs, best_labels)
}

with open(MAP_PATH, "w") as f:
    json.dump(event_type_map, f, indent=2)

print(f"Saved {len(event_type_map)} verb to label mappings to {MAP_PATH}")
print()
print("Sample mappings:")
for verb, label in list(event_type_map.items())[:8]:
    print(f"  {verb:25s}  :  {label}")

Loaded 30 cluster labels from data/cluster_labels.csv
Saved 3309 verb to label mappings to event_chains/event_type_map.json

Sample mappings:
  've                        :  Structural Experimentation
  -                          :  Structural Experimentation
  -influence                 :  Commerce & Value
  abandon                    :  Evolution & Hybridization
  abate                      :  Detailing & Enhancement
  abbreviate                 :  Mixing & Material Interaction
  abide                      :  Modification & Refinement
  abound                     :  Creation & Inspiration


## 4. Verb cluster diagnostics

In [13]:
# Cell 14: Compute each verb's distance to its assigned cluster centroid
centroids = best_km.cluster_centers_

distances = []
for verb, label, emb in zip(unique_verbs, best_labels, verb_embeddings):
    dist = float(np.linalg.norm(emb - centroids[label]))
    distances.append({
        "verb":             verb,
        "frequency":        verb_counter[verb],
        "assigned_cluster": CLUSTER_LABELS[int(label)],
        "centroid_distance": dist,
    })

diagnostic_df = pd.DataFrame(distances)
diagnostic_df = diagnostic_df.sort_values("centroid_distance", ascending=False)

diagnostic_df

,verb,frequency,assigned_cluster,centroid_distance
450,"cited""apparition",1,Commerce & Value,0.996941
651,createdextraordinary,1,Design Construction & Composition,0.976761
1515,integrate,19,Reduction & Retention,0.973216
522,commune,1,Reduction & Retention,0.971843
2200,react,3,Reinterpretation & Return,0.969555
...,...,...,...,...
2220,recast,7,Allusion & Comparison,0.648153
2298,rejuvenate,1,Allusion & Comparison,0.645369
2375,resurrect,4,Allusion & Comparison,0.643879
2242,recur,12,Allusion & Comparison,0.640054


In [14]:
# Cell 15: Save diagnostics table
diagnostic_df.to_csv(DIAG_PATH, index=False)
print(f"Saved {len(diagnostic_df)} rows to {DIAG_PATH}")

Saved 3309 rows to event_chains/domain_verb_diagnostics.csv


## 5. Per-review event sequences

In [15]:
# Cell 16: Map verbs to event types and save event_sequences.jsonl

sequences = []
for r in triples:
    events = []
    for t in r["triples"]:
        event_type = event_type_map.get(t["verb"], "UNMAPPED")
        events.append({
            "event":      event_type,
            "verb":       t["verb"],
            "is_passive": t["is_passive"],
        })
    sequences.append({
        "review_id": r["review_id"],
        "brand":     r["brand"],
        "year":      r["year"],
        "season":    r["season"],
        "author":    r["author"],
        "city":      r["city"],
        "events":    events,
    })

with open(SEQUENCE_PATH, "w") as f:
    for s in sequences:
        f.write(json.dumps(s) + "\n")

print(f"Saved {len(sequences)} event sequences to {SEQUENCE_PATH}")

Saved 6501 event sequences to checkpoints/event_sequences.jsonl


## 6. Sanity checks

In [ ]:
# Cell 17: Top-20 verb lemmas before and after abstraction

print("Top 20 verb lemmas before clustering")
for v, c in verb_counter.most_common(20):
    print(f"  {v:25s}  {c:>6,}  :  {event_type_map.get(v, 'UNMAPPED')}")

print()
event_counter = Counter()
for s in sequences:
    for e in s["events"]:
        event_counter[e["event"]] += 1

print(f"All event types after clustering ({len(event_counter)} total)")
for et, c in event_counter.most_common():
    print(f"  {c:>7,}  {et}")

In [29]:
# Cell 18: Passive vs. active breakdown overall and per event type

all_triples = [t for r in triples for t in r["triples"]]
n_passive = sum(t["is_passive"] for t in all_triples)
n_total   = len(all_triples)
print(f"Overall: {n_passive:,} passive / {n_total:,} total = {n_passive/n_total:.1%}")

# Per event type
et_passive = {}
et_total   = {}
for r in triples:
    for t in r["triples"]:
        et = event_type_map.get(t["verb"], "UNMAPPED")
        et_total[et]   = et_total.get(et, 0) + 1
        et_passive[et] = et_passive.get(et, 0) + int(t["is_passive"])

print("Passive rate per event type")
rows = [(et, et_passive[et], et_total[et], et_passive[et]/et_total[et])
        for et in et_total]
rows.sort(key=lambda x: x[3], reverse=True)
print(f"{'Event type':<38} {'Passive':>8} {'Total':>8} {'Rate':>6}")
for et, n_p, n_t, rate in rows:
    print(f"  {et:<36} {n_p:>8,} {n_t:>8,} {rate:>6.1%}")

Overall: 8,883 passive / 119,370 total = 7.4%
Passive rate per event type
Event type                              Passive    Total   Rate
  Action & Introduction                     720    2,469  29.2%
  Reinterpretation & Return                 612    2,144  28.5%
  Emphasis Through Contrast                  51      197  25.9%
  Physical Manipulation & Handling          121      522  23.2%
  Design Construction & Composition          94      407  23.1%
  Revelation & Exposure                     189      874  21.6%
  General Evaluation & Experience            93      453  20.5%
  Detailing & Enhancement                   152      745  20.4%
  Craftsmanship & Surface Work               56      317  17.7%
  Naming & Characterization                 165      947  17.4%
  Presentation & Debut                      355    2,151  16.5%
  Evolution & Hybridization                 128      791  16.2%
  Mixing & Material Interaction             108      736  14.7%
  Allusion & Comparison       

In [20]:
# Cell 19: Inspect reviews with empty or very short event sequences
short_reviews = [
    {"review_id": s["review_id"], "brand": s["brand"],
     "year": s["year"], "n_events": len(s["events"])}
    for s in sequences if len(s["events"]) < 2
]

print(f"Reviews with < 2 events: {len(short_reviews)} / {len(sequences)}")
if short_reviews:
    print(pd.DataFrame(short_reviews).to_string(index=False))

Reviews with < 2 events: 9 / 6501
 review_id                 brand  year  n_events
      4031    Christophe Lemaire  2011         1
      4521            John Rocha  2012         1
      5377         ZAC Zac Posen  2013         1
      5384 McQ Alexander McQueen  2013         1
      5469           Gary Graham  2013         1
      5643                   Sea  2014         1
      5729                  Daks  2014         1
      6028            James Long  2014         1
      6111                 Chloé  2014         1
